# Set 07 – Gaussian Naive Bayes

Naive Bayes kombiniert drei Ideen:

- Prior: Wie häufig ist eine Klasse vor Betrachtung der Merkmale?
- Likelihood: Wie typisch sind die beobachteten Merkmale für diese Klasse?
- Posterior: Wie wahrscheinlich ist die Klasse nach Betrachtung der Merkmale?

Gaussian Naive Bayes nimmt für numerische Merkmale je Klasse eine Normalverteilung an.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

rng = np.random.default_rng(42)

## 1. Kontrollierte Sensordaten

Klasse 0 beschreibt einen normalen Zustand, Klasse 1 einen auffälligen Zustand. Die beiden Merkmale besitzen je Klasse unterschiedliche Mittelwerte und Streuungen.

In [ ]:
n_normal = 420
n_alarm = 180

normal = pd.DataFrame({
    "temperatur_c": rng.normal(65, 5.5, n_normal),
    "vibration_mm_s": rng.normal(3.0, 0.7, n_normal),
    "klasse": 0,
})
alarm = pd.DataFrame({
    "temperatur_c": rng.normal(78, 7.0, n_alarm),
    "vibration_mm_s": rng.normal(5.2, 1.0, n_alarm),
    "klasse": 1,
})

daten = pd.concat([normal, alarm], ignore_index=True)
daten = daten.sample(frac=1, random_state=42).reset_index(drop=True)

print(daten["klasse"].value_counts())
print(daten["klasse"].value_counts(normalize=True).round(3))
display(daten.groupby("klasse").agg(["mean", "std"]).round(2))

## 2. Daten nach Klasse visualisieren

Gaussian Naive Bayes lernt für jedes Merkmal und jede Klasse eine eigene Normalverteilung. Die Punktwolke und Histogramme machen diese Verteilungen sichtbar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for klasse, name, farbe in [
    (0, "normal", "#4C78A8"),
    (1, "auffällig", "#E45756"),
]:
    teil = daten[daten["klasse"] == klasse]
    axes[0].scatter(
        teil["temperatur_c"], teil["vibration_mm_s"],
        label=name, alpha=0.55, color=farbe
    )
    axes[1].hist(teil["temperatur_c"], bins=22, alpha=0.55, label=name, color=farbe)
    axes[2].hist(teil["vibration_mm_s"], bins=22, alpha=0.55, label=name, color=farbe)

axes[0].set_xlabel("Temperatur in °C")
axes[0].set_ylabel("Vibration in mm/s")
axes[0].set_title("Gemeinsame Ansicht")
axes[1].set_title("Temperatur je Klasse")
axes[2].set_title("Vibration je Klasse")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Training und Test trennen

stratify erhält die Klassenanteile. Das Modell lernt Priors, Mittelwerte und Varianzen ausschließlich aus den Trainingsdaten.

In [ ]:
X = daten[["temperatur_c", "vibration_mm_s"]]
y = daten["klasse"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modell = GaussianNB()
modell.fit(X_train, y_train)

## 4. Gelernte Parameter

class_prior_ enthält die Klassenhäufigkeiten. theta_ enthält die Mittelwerte und var_ die Varianzen je Klasse und Merkmal.

In [ ]:
parameter = []
for klassen_index, klasse in enumerate(modell.classes_):
    for merkmals_index, merkmal in enumerate(X.columns):
        parameter.append({
            "Klasse": klasse,
            "Merkmal": merkmal,
            "Mittelwert": modell.theta_[klassen_index, merkmals_index],
            "Varianz": modell.var_[klassen_index, merkmals_index],
        })

print("Klassen:", modell.classes_)
print("Gelernte Priors:", modell.class_prior_.round(3))
display(pd.DataFrame(parameter).round(3))

## 5. Bayes-Rechnung für einen einzelnen Fall

Für jede Klasse wird berechnet:

Prior mal Likelihood Temperatur mal Likelihood Vibration

Nach dem Normieren entstehen Posterior-Wahrscheinlichkeiten. Mit nur zwei Merkmalen können wir dies direkt nachrechnen.

In [ ]:
def normaldichte(x, mittelwert, varianz):
    faktor = 1 / np.sqrt(2 * np.pi * varianz)
    exponent = np.exp(-((x - mittelwert) ** 2) / (2 * varianz))
    return faktor * exponent

fall = pd.DataFrame({
    "temperatur_c": [75.0],
    "vibration_mm_s": [4.8],
})

unnormiert = []
for klassen_index, klasse in enumerate(modell.classes_):
    score = modell.class_prior_[klassen_index]
    for merkmals_index, merkmal in enumerate(X.columns):
        score *= normaldichte(
            fall.iloc[0, merkmals_index],
            modell.theta_[klassen_index, merkmals_index],
            modell.var_[klassen_index, merkmals_index],
        )
    unnormiert.append(score)

posterior_manuell = np.asarray(unnormiert) / np.sum(unnormiert)
posterior_sklearn = modell.predict_proba(fall)[0]

vergleich = pd.DataFrame({
    "Klasse": modell.classes_,
    "manuell": posterior_manuell,
    "scikit_learn": posterior_sklearn,
})
display(vergleich.round(5))
print("Vorhersage:", modell.predict(fall)[0])

## 6. Testdaten bewerten

Auch bei ungleichen Klassen betrachten wir Recall und Konfusionsmatrix statt nur Accuracy.

In [ ]:
y_pred = modell.predict(X_test)
bericht = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        target_names=["normal", "auffällig"],
        output_dict=True,
        zero_division=0,
    )
).T
display(bericht.round(3))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["normal", "auffällig"], cmap="Blues"
)
plt.title("Gaussian Naive Bayes")
plt.show()

## 7. Entscheidungsfläche

Die Grenze entsteht aus den erlernten Klassenverteilungen. Sie muss nicht zwingend eine gerade Linie sein.

In [ ]:
t_werte = np.linspace(X["temperatur_c"].min() - 3, X["temperatur_c"].max() + 3, 250)
v_werte = np.linspace(X["vibration_mm_s"].min() - 0.5, X["vibration_mm_s"].max() + 0.5, 250)
gitter_t, gitter_v = np.meshgrid(t_werte, v_werte)
gitter = pd.DataFrame({
    "temperatur_c": gitter_t.ravel(),
    "vibration_mm_s": gitter_v.ravel(),
})
proba = modell.predict_proba(gitter)[:, 1].reshape(gitter_t.shape)

fig, ax = plt.subplots(figsize=(9, 6))
flaeche = ax.contourf(gitter_t, gitter_v, proba, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.65)
ax.contour(gitter_t, gitter_v, proba, levels=[0.5], colors="black", linewidths=2)
ax.scatter(X_test["temperatur_c"], X_test["vibration_mm_s"], c=y_test, cmap="bwr", edgecolor="white")
ax.set_xlabel("Temperatur in °C")
ax.set_ylabel("Vibration in mm/s")
ax.set_title("P(auffällig) und Entscheidungsgrenze")
fig.colorbar(flaeche, ax=ax, label="P(Klasse 1)")
plt.show()

## Grenzen

- Die Merkmale werden innerhalb jeder Klasse als unabhängig behandelt.
- Für jedes numerische Merkmal wird eine Normalverteilung angenommen.
- Wahrscheinlichkeiten können sehr sicher wirken, obwohl Annahmen verletzt sind.
- Gute Modellgüte muss auf unbekannten Daten geprüft werden.
- Für Worthäufigkeiten verwenden wir später Multinomial Naive Bayes statt GaussianNB.